# Synapse Foundations Lab

Executable local notebook for the mathematical foundations used by Synapse projects: tokenization, vector similarity, metrics, optimization intuition, calibration, and retrieval evaluation. It has no network dependency.

In [ ]:
import math
import re
from collections import Counter

def tokenize(text):
    return re.findall(r"[a-z0-9_]+", text.lower())

def tf_vector(text):
    counts = Counter(tokenize(text))
    norm = math.sqrt(sum(value * value for value in counts.values())) or 1.0
    return {token: value / norm for token, value in counts.items()}

def cosine(left, right):
    return sum(left.get(token, 0.0) * right.get(token, 0.0) for token in set(left) | set(right))

documents = {
    "rag": "RAG corporativo combina recuperacao, citacoes, avaliacao e controle de alucinacao.",
    "ml": "Machine learning exige contrato de dados, baseline, metricas, drift e monitoramento.",
    "cost": "O Synapse usa Ollama local first e ativa poucos agentes para reduzir custo por token."
}
query = "como reduzir custo de tokens com agentes locais"
query_vector = tf_vector(query)
scores = sorted(
    ((doc_id, cosine(query_vector, tf_vector(text))) for doc_id, text in documents.items()),
    key=lambda item: item[1],
    reverse=True,
)
scores


In [ ]:
def regression_metrics(expected, predicted):
    errors = [p - y for y, p in zip(expected, predicted)]
    mae = sum(abs(error) for error in errors) / len(errors)
    rmse = math.sqrt(sum(error * error for error in errors) / len(errors))
    return {"mae": mae, "rmse": rmse}

def classification_metrics(expected, predicted):
    tp = sum(1 for y, p in zip(expected, predicted) if y == 1 and p == 1)
    fp = sum(1 for y, p in zip(expected, predicted) if y == 0 and p == 1)
    fn = sum(1 for y, p in zip(expected, predicted) if y == 1 and p == 0)
    tn = sum(1 for y, p in zip(expected, predicted) if y == 0 and p == 0)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"accuracy": (tp + tn) / len(expected), "precision": precision, "recall": recall, "f1": f1}

regression_metrics([10, 20, 30], [12, 18, 33]), classification_metrics([1, 0, 1, 0], [1, 0, 0, 0])


In [ ]:
def gradient_descent(start=0.0, target=3.0, learning_rate=0.2, steps=12):
    value = start
    history = []
    for step in range(steps):
        gradient = 2 * (value - target)
        value -= learning_rate * gradient
        history.append({"step": step + 1, "value": value, "loss": (value - target) ** 2})
    return history

gradient_descent()[-3:]


In [ ]:
def retrieval_eval(relevant_ids, ranked_ids, k=3):
    retrieved = ranked_ids[:k]
    hits = [doc_id for doc_id in retrieved if doc_id in relevant_ids]
    precision_at_k = len(hits) / k
    recall_at_k = len(hits) / len(relevant_ids) if relevant_ids else 0.0
    return {"precision_at_k": precision_at_k, "recall_at_k": recall_at_k, "hits": hits}

retrieval_eval({"cost"}, [doc_id for doc_id, _score in scores], k=2)
